# 25_01 — Simulación y preprocesamiento · **Escenario G: tres regímenes con tendencia multimodal**

*tres ramas, cada una con su propia FORMA de tendencia*

Paso **1 de 3** del ciclo `Python → MATLAB → Python`. Las celdas marcadas
**`[CONFIG]`** son las únicas que se tocan al cambiar de experimento.

## Qué es este escenario

**Tres regímenes, y cada uno con su propia forma de tendencia.** El generador es

$$X_t(\tau) = \mu(\tau) + d_{S_t}\,b_{S_t}(t)\,g(\tau) + f_{S_t}\,\Psi Y_{t-1}(\tau) + \varepsilon_t(\tau),$$

con $S_t\in\{0,1,2\}$ sorteado por un **probit ordenado** sobre
$z_{t-1}=\langle Y_{t-1},e\rangle$ —el mismo mecanismo del Algoritmo 3, con dos
cortes en vez de uno— y con

| rama | operador $f$ | deriva $d$ | **forma de la tendencia** | lectura |
|---|---|---|---|---|
| 0 | $+0.9$ | $+1$ | **cuadrática** | persiste y **acelera** hacia arriba |
| 1 | $0$ | $0$ | sinusoidal | se olvida del pasado y **oscila** |
| 2 | $-0.9$ | $-1$ | **logarítmica** | invierte y **se aplana** hacia abajo |

La novedad respecto de los Escenarios E y F: allí las dos ramas compartían la
forma de la tendencia y sólo diferían en un escalar. **Aquí las tres modas no
sólo se separan: lo hacen siguiendo trayectorias de forma distinta**, de modo
que la distancia entre ellas cambia de orden a lo largo de la serie. Una rama
que acelera y otra que se aplana se cruzan y se vuelven a separar.

### Por qué el coeficiente de Sarle **baja** aquí, y no es un fracaso

Con tres modas aproximadamente equiespaciadas la moda central **rellena el
hueco** entre las extremas y la densidad se aplana: el coeficiente de Sarle,
que detecta *bi*modalidad, baja aunque la multimodalidad sea mayor. Por eso el
diagnóstico agrega **`modas_efectivas`** $=1/\sum_j p_j^2$ —el número efectivo
de componentes activas por origen—, y con $J=3$ la lectura correcta es el par
(modas efectivas, separación máxima), con el Sarle en segundo plano.

---

## La familia con tendencia: dónde encaja esta corrida

Los siete escenarios de `pipelines/sim_escenario_T.py` comparten esquema de
observación, operador base, innovación, priors y `mcmc_config` — todo lo que no
sea el mecanismo. Sólo así las diferencias son entre generadores.

| corrida | no linealidad | tendencia |
|---|---|---|
| 19 · C | interacciones intra-curva | lineal |
| 22 · D | interacciones intra-curva | cuadrática |
| 23 · E | 2 regímenes | lineal por régimen |
| 24 · F | 2 regímenes | cuadrática por régimen |
| **25 · G** | **3 regímenes** | **una forma distinta por régimen** |
| **26 · H** | 2 regímenes | **sinusoidal y volátil** |
| **27 · I** | **3 regímenes** | **por tramos: cuadrática → log → sinusoidal** |

**Ninguno es estacionario**, y eso trae de vuelta la lección de la corrida 17:
el FPCA y el estandarizador se ajustan con el bloque de entrenamiento y **dejan
de ser válidos en el de prueba**. Parte del error de test es de *vigencia del
centrado* y no de capacidad predictiva, y por eso el diagnóstico reporta todo
**por duplicado**, en bruto y sobre el proceso destendenciado.


## 1. Imports y rutas

In [ ]:
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Generadores y contrato de artefactos
from model_psbp_fd.pipelines import (
    ConfigEscenarioT, generar_escenario_T, guardar_escenario,
    guardar_curvas, guardar_representacion, guardar_fpca,
    guardar_estandarizador, guardar_datasets_ar,
    guardar_hiperparametros, guardar_config_evaluacion,
    verificar_contrato,
)
# Preprocesamiento funcional
from model_psbp_fd.functions_models import (
    FunctionalRepresentation, FPCA_L2, base_en_grilla, DataStandardizer,
)
from model_psbp_fd.fit import tabla_baselines
from model_psbp_fd.utils import get_project_root
from model_psbp_fd.graphics import (
    plot_empirical_sample, plot_mean_and_variance, plot_fts_empirical,
    plot_fts_functional, plot_diagnostico_estandarizacion, plot_fpca_scree,
    plot_seleccion_basis, plot_rezagos_heatmap,
)

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

### 1.1 `[CONFIG]` Identificación del experimento

`ESCENARIO_ID` es una **cadena** (`"B"`), no un entero: el `psbp_fd_iteracion.m`
de esta carpeta usa `%s` en su `sprintf` por ese motivo, igual que el de la
corrida 17.

`M_FPCA` vive aquí y no en §3.4 porque forma parte del `EXPERIMENT_ID`, y las
cinco rutas del contrato se construyen en la celda siguiente — antes de que
exista el objeto FPCA. La convención del identificador es

    <basename>_<escenario>_r<réplica a 2 dígitos>_m<M a 2 dígitos>

de modo que cada punto del barrido en `M` escribe sus propios datos, trazas y
reportes, y ninguno pisa a los demás.

In [ ]:
PROJECT_ROOT = get_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BASENAME     = "escenario"
ESCENARIO_ID = "G"    # escenario de diagnóstico; NO es un Algoritmo del anexo
REPLICA_ID   = 1      # réplica Monte Carlo; eje del barrido en la Etapa D
SEED         = 41232  # semilla base; MATLAB la LEE de hyperparameters.json

# ── [BARRIDO] Componentes FPCA retenidas ─────────────────────────────────
# M entra DOS VECES en el modelo: como número de mezclas y como dimensión
# p = q·M del predictor de los pesos probit. Por eso es un eje del estudio y no
# un detalle de preprocesamiento, y por eso viaja en el EXPERIMENT_ID: cada
# valor de M escribe sus propios artefactos y no pisa los de los demás.
#
# La regla declarada es la primera M con varianza acumulada ≥ 95 %; los valores
# por encima y por debajo se corren para medir la sensibilidad. §3.4 contrasta
# este valor con el que da la regla y verifica que no exceda el K disponible.
#
# Al cambiarlo hay que cambiarlo TAMBIÉN en 25_03, 25_04, 25_05 y en
# psbp_fd_iteracion.m: los cinco arman el mismo EXPERIMENT_ID a mano.

# Usaremos (1, 2, 3). El punto M=1 es el que la portada predice que debe
# fallar: con una sola componente el modelo no observa la dirección de
# conmutación, que carga sobre la SEGUNDA. No es un punto de relleno del
# barrido, es la mitad del experimento.
M_FPCA = 2

EXPERIMENT_ID = f"{BASENAME}_{ESCENARIO_ID}_r{REPLICA_ID:02d}_m{M_FPCA:02d}"

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"EXPERIMENT_ID : {EXPERIMENT_ID}")
print(f"Escenario {ESCENARIO_ID} · réplica {REPLICA_ID} · M {M_FPCA} · seed base {SEED}")

In [ ]:
# Las cinco rutas del contrato. No existe un config_paths en Python: cada
# notebook lo arma aquí y config_paths.m replica las mismas del lado MATLAB.
PATHS = {
    "raw":          PROJECT_ROOT / "data" / "simulaciones" / "raw" / EXPERIMENT_ID,
    "functional":   PROJECT_ROOT / "data" / "simulaciones" / "processed" / "functional" / EXPERIMENT_ID,
    "predict":      PROJECT_ROOT / "data" / "simulaciones" / "processed" / "predict" / EXPERIMENT_ID,
    "out_report":   PROJECT_ROOT / "reports" / "simulaciones" / EXPERIMENT_ID,
    "out_artefact": PROJECT_ROOT / "artefact" / "simulaciones" / EXPERIMENT_ID,
}
for nombre, ruta in PATHS.items():
    ruta.mkdir(parents=True, exist_ok=True)
    print(f"  {nombre:12s} → {ruta}")

## 2. Simulación

### 2.1 `[CONFIG]` Parámetros del generador

El esquema de observación y la parte lineal son **los mismos que en las
corridas 18 a 24** —`L=75`, `T=400`, `PROP_TRAIN=0.70`, `sigma_obs=0.25`,
`mu = sin(2 pi tau)`, `gamma=0.30`, `hs_norm=0.70`, `ell=0.5`— y no se
tocan: sólo así las diferencias entre corridas son entre generadores.

Lo que distingue a esta corrida es el bloque marcado abajo. La celda §2.3
verifica que el generador hizo lo que se le pidió **antes** de gastar MCMC
en él.

In [ ]:
# -- Parámetros fijos del estudio (comunes a todas las corridas) -------------
L_GRILLA   = 75     # puntos de la grilla regular tau_1=0 ... tau_L=1
T_CURVAS   = 400    # curvas retenidas tras el calentamiento
PROP_TRAIN = 0.70   # proporción del bloque de entrenamiento  -> T0 = 280
SIGMA_OBS  = 0.25   # desviación del ruido de medición

def media_senoidal(tau):
    """mu(tau) = sin(2 pi tau). La media vigente del estudio (tab:ane_esquema)."""
    return np.sin(2.0 * np.pi * tau)

SIM_CFG = ConfigEscenarioT(
    # Esquema de observación - idéntico a las corridas 18 y 20
    L         = L_GRILLA,
    T         = T_CURVAS,
    burn_in   = 200,
    sigma_obs = SIGMA_OBS,
    R         = 1,            # una réplica por corrida; el barrido usa REPLICA_ID
    seed      = SEED,
    media_fn  = media_senoidal,
    # Parte lineal - la del Algoritmo 1, sin tocar
    gamma     = 0.30,
    hs_norm   = 0.70,
    sigma_eps = 1.0,
    ell       = 0.5,
    # ── Eje 1: la TENDENCIA, una FORMA por rama ──────────────────────────
    deriva          = 3.0,
    forma_tendencia = "lineal",          # sólo respaldo: manda `formas_regimen`
    inclinacion     = 0.0,
    # ── Eje 2: TRES regímenes ────────────────────────────────────────────
    mecanismo         = "mezcla",
    factores_operador = (0.9, 0.0, -0.9),   # persiste / se olvida / invierte
    derivas_regimen   = (1.0, 0.0, -1.0),   # sube / plana / baja
    formas_regimen    = ("cuadratica",      # rama 0: acelera
                         "sinusoidal",      # rama 1: oscila
                         "logaritmica"),    # rama 2: se aplana
    umbrales          = (-0.17, 0.17),      # dos cortes => tres ramas
    nitidez           = 4.0,
    # Referencia sólo para el diagnóstico (no afecta a la generación)
    prop_train_referencia = PROP_TRAIN,
)

for k, v in SIM_CFG.to_dict().items():
    print(f"  {k:<24}: {v}")

# Los dos ejes del factorial deben ser exactamente los declarados: si se cambian
# aquí sin cambiar el EXPERIMENT_ID, esta corrida deja de ser comparable con las
# otras tres y la atribución de efectos se pierde en silencio.
assert SIM_CFG.mecanismo == "mezcla", \
    "Los ejes del generador no coinciden con los que declara el EXPERIMENT_ID."


### 2.2 Generación

In [ ]:
salida = generar_escenario_T(SIM_CFG)

REPLICA_IDX = REPLICA_ID - 1
X_raw  = salida.observaciones[REPLICA_IDX]   # (T, G) OBSERVADA - alimenta la estimación
X_true = salida.curvas[REPLICA_IDX]          # (T, G) VERDADERA - objetivo de evaluación
grilla = salida.grilla
T, G   = X_raw.shape

DIAG = salida.diagnostico

print(f"Observadas {X_raw.shape} · verdaderas {X_true.shape} · grilla {grilla.shape}")
print(f"Ruido de medición efectivo: sd(X_raw - X_true) = {(X_raw - X_true).std():.4f}"
      f"   (nominal {SIGMA_OBS})")
print("\nControl de calidad del generador:")
for k, v in DIAG.items():
    print(f"  {k:38s} = {v}")


### 2.3 Verificación del mecanismo — **la celda que decide si el escenario sirve**

Antes de ajustar nada se comprueba que el generador hizo lo que se le pidió.
Aquí la verificación es, además, el **resultado central** del escenario.

Tres bloques y cinco `assert`:

1. **La tendencia se inyectó con la magnitud pedida.** Se regresa el nivel
   puntual de cada curva sobre $b(t)$ y la pendiente debe recuperar
   $\overline{g(\tau)}$ (ponderado por la ocupación de las ramas si hay
   régimen). El error estándar se corrige por autocorrelación: el residuo de esa
   regresión es la componente $Y$ promediada sobre $\tau$, que es fuertemente
   autocorrelacionada, y el error estándar de MCO la subestima.
2. **La componente sin tendencia es estable.** El escenario no es estacionario
   —la tendencia lo impide— pero $Y_t$ debe serlo. Si
   `razon_varianza_Y_mitades` se aleja de 1, la no linealidad desestabilizó la
   recursión y las conclusiones serían sobre una explosión, no sobre la no
   linealidad.
3. **Hay una brecha entre el mejor lineal y el oráculo, y no es sólo la
   tendencia.** Las dos cifras: en bruto y destendenciada. La segunda es la que
   dice si el escenario tiene contenido dinámico o si el VAR va a parecer
   excelente por la razón equivocada.


In [ ]:
# -- 1. La tendencia se inyectó como se pidió -------------------------------
print(f"pendiente recuperada        : {DIAG['pendiente_recuperada']:+.4f}   "
      f"(esperada {DIAG['pendiente_esperada']:+.4f})")
print(f"error estándar (MCO)        : {DIAG['pendiente_ee']:.4f}")
print(f"error estándar (autocorr.)  : {DIAG['pendiente_ee_autocorr']:.4f}   "
      f"<- acf1 del residuo = {DIAG['acf1_residuo_tendencia']:.3f}")
print(f"desvío en EE corregidos     : {DIAG['desvio_pendiente_en_ee']:.2f}")
print(f"\nb(T0) = {DIAG['b_en_T0']:.3f}   b(T) = {DIAG['b_en_T']:.3f}   "
      f"deriva total = {DIAG['deriva_total']:.2f}")
print(f"desfase train->test         : {DIAG['desfase_train_test']:+.3f}  =  "
      f"{DIAG['desfase_en_sd']:.2f} sd del proceso")
print(f"deriva total / sd           : {DIAG['razon_deriva_sd']:.2f} sd")

assert DIAG["desvio_pendiente_en_ee"] < 3.0, (
    f"La pendiente recuperada ({DIAG['pendiente_recuperada']:.4f}) dista "
    f"{DIAG['desvio_pendiente_en_ee']:.1f} errores estándar de la esperada "
    f"({DIAG['pendiente_esperada']:.4f}): la tendencia NO se inyectó como se pidió.")
assert DIAG["razon_deriva_sd"] > 1.0, (
    "La deriva es menor que una desviación del proceso: el efecto de la "
    "tendencia no será separable del ruido con R=1. Subir `deriva`.")

# -- 2. La componente sin tendencia es estable ------------------------------
print(f"\nvar(Y) 1a mitad / 2a mitad  : {DIAG['var_Y_primera_mitad']:.4f} / "
      f"{DIAG['var_Y_segunda_mitad']:.4f}   razón = "
      f"{DIAG['razon_varianza_Y_mitades']:.3f}")
assert 0.5 < DIAG["razon_varianza_Y_mitades"] < 2.0, (
    "La varianza de la componente sin tendencia cambia demasiado entre mitades: "
    "la recursión no es estable y las conclusiones serían sobre una explosión.")

# -- 3. La brecha entre el mejor lineal y el oráculo ------------------------
print(f"\n{'':28s}{'en bruto':>12s}{'destendenciado':>18s}")
print(f"{'R2 mejor predictor lineal':28s}"
      f"{DIAG['r2_lineal_fuera_de_muestra']:>12.4f}"
      f"{DIAG['r2_lineal_destendenciado']:>18.4f}")
print(f"{'R2 oráculo (media cond.)':28s}"
      f"{DIAG['r2_oraculo_fuera_de_muestra']:>12.4f}"
      f"{DIAG['r2_oraculo_destendenciado']:>18.4f}")
print(f"{'brecha':28s}{DIAG['brecha_oraculo_lineal']:>12.4f}"
      f"{DIAG['brecha_oraculo_lineal_destendenciada']:>18.4f}")
print(f"\nR2 lineal DENTRO de muestra : "
      f"{DIAG['r2_lineal_dentro_de_muestra']:+.4f}   <- inflado por sobreajuste")
print(f"acf1 puntual de la serie    : {DIAG['acf1_media']:+.4f}   "
      "<- con tendencia, alta por construcción")

assert DIAG["brecha_oraculo_lineal_destendenciada"] > 0.03, (
    "Sin la tendencia no queda brecha entre el mejor lineal y el oráculo: todo "
    "lo que el escenario aporta sería la tendencia, y un des-tendenciado previo "
    "lo resolvería. El escenario no discrimina modelos.")

# -- 4. La mezcla: ocupación, ambigüedad y modas que se separan -------------
print(f"\nproporción de la rama 0     : {DIAG['proporcion_regimen_0']:.3f}")
print(f"duración media de racha     : {DIAG['duracion_media_racha']:.2f} períodos")
print(f"nitidez en sd(z)            : {DIAG['nitidez_en_sd_z']:.2f}")
print(f"orígenes ambiguos           : {DIAG['fraccion_origenes_ambiguos']:.1%}"
      f"  ({DIAG['n_origenes_ambiguos']} de {T})")

assert DIAG["fraccion_origenes_ambiguos"] > 0.15, (
    "Menos del 15 % de los orígenes son ambiguos: la conmutación es casi "
    "determinista y no quedan orígenes con varias modas. Bajar `nitidez`.")

print("\nSeparación entre las modas extremas de la ley condicional:")
print(f"  parte dinámica (constante en t) : "
      f"{DIAG['separacion_modas_dinamica_L2']:.3f}")
print(f"  parte de tendencia en T0        : "
      f"{DIAG['separacion_modas_tendencia_en_T0']:.3f}")
print(f"  parte de tendencia en T         : "
      f"{DIAG['separacion_modas_tendencia_en_T']:.3f}")
print(f"  TOTAL en sd de la innovación    : "
      f"{DIAG['separacion_total_en_sd_T0']:.2f} (T0)  ->  "
      f"{DIAG['separacion_total_en_sd_T']:.2f} (T)")

assert DIAG["separacion_total_en_sd_T"] > DIAG["separacion_total_en_sd_T0"], (
    "Las modas no se separan más con el tiempo: `derivas_regimen` no está "
    "haciendo su trabajo y el escenario pierde su rasgo propio.")

# -- 5. Bimodalidad de la ley condicional VERDADERA (referencia oracle) -----
print("\ncoeficiente de Sarle EXACTO de la ley condicional verdadera")
print("(referencias: uniforme 0.5556 · gaussiana 0.3333)")
print(f"  ambiguos, bloque de TRAIN  : {DIAG['sarle_oraculo_ambiguos_train']:.4f}")
print(f"  ambiguos, bloque de TEST   : {DIAG['sarle_oraculo_ambiguos_test']:.4f}"
      "   <- crece con t: es el rasgo del escenario")
print(f"  deterministas, TEST        : {DIAG['sarle_oraculo_deterministas_test']:.4f}")

assert DIAG["sarle_oraculo_ambiguos_test"] > DIAG["sarle_oraculo_ambiguos_train"], (
    "La bimodalidad de la ley VERDADERA no crece entre train y test: sin eso, "
    "la sección 9.1 del _04 no tiene nada que detectar.")


# -- 6. Las tres ramas y sus tres formas de tendencia -----------------------
print("\nforma de tendencia por rama :", DIAG["formas_regimen"])
print("cortes del probit ordenado  :", np.round(DIAG["cortes_probit"], 3))
print("ocupación por rama          :", np.round(DIAG["proporcion_por_regimen"], 3))
print(f"ocupación mínima            : {DIAG['proporcion_regimen_minima']:.3f}")
print(f"modas efectivas (media/máx) : {DIAG['modas_efectivas_media']:.3f} / "
      f"{DIAG['modas_efectivas_maxima']:.3f}   (de {DIAG['n_regimenes']} posibles)")

assert DIAG["proporcion_regimen_minima"] > 0.10, (
    "Una de las tres ramas ocurre en menos del 10 % de los períodos: no habrá "
    "orígenes suficientes para estratificar por ella. Acercar los `umbrales`.")
assert DIAG["modas_efectivas_media"] > 1.3, (
    "El número efectivo de componentes activas es cercano a 1: el probit está "
    "casi determinista y la mezcla no se distingue de un solo régimen.")

print("\nOJO con la lectura del coeficiente de Sarle en este escenario: con TRES\n"
      "modas aproximadamente equiespaciadas la central rellena el hueco entre las\n"
      "extremas y el coeficiente BAJA, aunque la multimodalidad sea mayor. La\n"
      "cifra que manda aquí es `modas_efectivas` junto con la separación máxima.")

print("\n[OK] mecanismo verificado: hay tendencia, la recursión es estable y las "
      "modas\n     se separan a lo largo del tiempo.")


In [ ]:
# Configuración de simulación + escenario completo en .npz
simulation_config = {
    "sim_params":    salida.config.to_dict(),
    "diagnostico":   DIAG,
    "replica_idx":   REPLICA_IDX,
    "experiment_id": EXPERIMENT_ID,
    "escenario_id":  ESCENARIO_ID,     # cadena "B", no entero; el .m usa %s
    "replica_id":    int(REPLICA_ID),
    "seed":          SEED,
    "T": int(T), "G": int(G),
}
with open(PATHS["raw"] / "simulation_config.json", "w", encoding="utf-8") as f:
    json.dump(simulation_config, f, indent=2, ensure_ascii=False)

_npz = guardar_escenario(salida, str(PATHS["raw"] / f"escenario_{ESCENARIO_ID}"),
                         incluir_curvas=True, incluir_internos=True)
print(f"[raw] simulation_config.json  ·  {_npz}")

### 2.4 Persistencia del estado verdadero

El **estado verdadero** aquí es el trío $(S_t,\;p_t,\;b(t))$ —rama vigente,
probabilidad con que se sorteó y deriva acumulada—, y se persiste igual que en
la corrida 18.

La diferencia con la 18, y es la que da sentido a este escenario: `separacion`
—la distancia entre las dos medias condicionales— **crece con $t$**, porque a la
parte dinámica se le suma $\lvert d_0-d_1\rvert\,b(t)\lVert g\rVert$. Un origen
ambiguo del final del bloque de prueba es mucho más bimodal que uno del
principio del entrenamiento, y `_04 §9.1` compara exactamente eso.

`ambiguo` y `bimodal` no son lo mismo: `bimodal` exige además que la separación
supere la desviación de la innovación, que es la condición para que las dos
modas se distingan.

A diferencia de la corrida 16, **el estrato no coincide con la partición
train/test**: la ambigüedad depende del estado rezagado y está repartida a lo
largo de toda la serie. La celda lo verifica.


In [ ]:
T0_prov = int(np.floor(PROP_TRAIN * T_CURVAS))   # provisional; se recalcula en 2.7

regimen  = salida.internos["regimenes"][REPLICA_IDX]            # (T,) 0..J-1
PR       = salida.internos["prob_regimenes"][REPLICA_IDX]       # (T, J)
z_lag    = salida.internos["proyeccion_estado"][REPLICA_IDX]    # (T,) z_{t-1}
M_ORACLE = salida.internos["media_condicional"][REPLICA_IDX]    # (T, G)
TEND     = salida.internos["tendencia"][REPLICA_IDX]            # (T, G) realizada
perfiles = salida.internos["perfiles_regimen"]                  # (J, T)
v_t      = salida.internos["factor_volatil"]                    # (T,)
g_tau    = salida.internos["forma_tendencia_tau"]
Psi_gen  = salida.internos["operador"]
w_quad   = salida.internos["pesos_cuadratura"]
J_REG    = PR.shape[1]

# Separación entre las modas extremas, origen por origen. La parte dinámica no
# depende de t; la de tendencia sí, y con una forma por rama cada par se separa
# a su propio ritmo — por eso se toma el máximo menos el mínimo sobre ramas.
Y_lag    = (X_true - TEND - salida.media)[:-1]
arrastre = np.vstack([np.full((1, G), np.nan), Y_lag @ Psi_gen.T])
df_op    = float(np.max(SIM_CFG.factores_operador) - np.min(SIM_CFG.factores_operador))
sep_din  = df_op * np.sqrt(np.sum(w_quad * arrastre ** 2, axis=1))
d_reg    = np.asarray(SIM_CFG.derivas_regimen, dtype=float)
B        = SIM_CFG.deriva * (d_reg[:, None] * perfiles) * v_t[None, :]   # (J, T)
sep_tend = (B.max(axis=0) - B.min(axis=0)) * float(np.sqrt(np.sum(w_quad * g_tau ** 2)))
separacion = sep_din + sep_tend
sd_innov = float(np.sqrt(np.sum(
    w_quad * np.diag(salida.internos["cov_innovacion"]))))

p_max    = PR.max(axis=1)
ambiguo  = p_max < 0.75          # ninguna rama domina; con J=2 es p in [.25,.75]
bimodal  = ambiguo & (separacion > sd_innov)
modas_ef = 1.0 / np.sum(PR ** 2, axis=1)

estado = pd.DataFrame({
    "t":             np.arange(1, T + 1),
    "regimen":       regimen.astype(int),
    **{f"p_regimen_{j}": PR[:, j] for j in range(J_REG)},
    "p_max":         p_max,
    "modas_efectivas": modas_ef,
    "z_lag":         z_lag,
    "ambiguo":       ambiguo.astype(int),
    "b_t_realizada": B[regimen, np.arange(T)],
    "factor_volatil": v_t,
    "sep_dinamica":  sep_din,
    "sep_tendencia": sep_tend,
    "separacion":    separacion,
    "sep_en_sd":     separacion / sd_innov,
    "bimodal":       bimodal.astype(int),
    "nivel_curva":   X_true.mean(axis=1),
    "bloque":        np.where(np.arange(1, T + 1) <= T0_prov, "train", "test"),
})
estado.to_csv(PATHS["out_report"] / "10_estado_signo_tendencia.csv", index=False)

# El régimen debe seguir a sus propias probabilidades; si no, hay un desfase de
# índices y toda la estratificación de _04 mediría otra cosa.
dom = PR.argmax(axis=1)
acierto = float((regimen[p_max > 0.75] == dom[p_max > 0.75]).mean())
print(f"régimen = rama dominante donde p_max>0.75 : {acierto:.3f}   (debe ser ~ 1)")
assert acierto > 0.85, \
    "El régimen no sigue a sus propias probabilidades: hay un desfase de índices."

print(f"\n[report] 10_estado_signo_tendencia.csv  ({len(estado)} filas · "
      f"{J_REG} ramas)")
print(estado.groupby("bloque")[["ambiguo", "bimodal", "modas_efectivas",
                                "sep_en_sd"]].mean().to_string())

sep_train = float(estado.loc[estado.bloque == "train", "sep_en_sd"].mean())
sep_test  = float(estado.loc[estado.bloque == "test",  "sep_en_sd"].mean())
print(f"\nseparación media entre modas: {sep_train:.2f} sd (train)  ->  "
      f"{sep_test:.2f} sd (test)")

# Los estratos no deben coincidir con la partición: si coincidieran, la
# cobertura condicional de _04 §9 mediría lo mismo que la comparación
# train/test de §4, que es el defecto declarado de la corrida 16.
frac_test_amb = float((estado.loc[estado.ambiguo == 1, "t"] > T0_prov).mean())
print(f"fracción de los orígenes ambiguos que cae en PRUEBA: {frac_test_amb:.1%}"
      "   (0.30 sería lo neutral)")
assert 0.05 < frac_test_amb < 0.75, (
    "Los orígenes ambiguos se concentran en un bloque: el estrato de la sección "
    "9 del _04 mediría lo mismo que la comparación train/test.")


### 2.5 Visualización de los datos

In [ ]:
highlight_idx = [0, 1, T // 2, T - 1]

plot_fts_empirical(
    X_raw, grilla, highlight_idx=highlight_idx, separator_every=5,
    title=f"FAR con signo conmutado — {T} curvas observadas (escala original)",
    save_path=str(PATHS["out_report"] / "01_fts_empirica_raw.png"))
plt.show()

plot_empirical_sample(
    X_raw, grilla, sample_idx=[0, 40, 80, 200, T - 1],
    title="Muestra de 5 curvas observadas",
    save_path=str(PATHS["out_report"] / "02_muestra_empirica_raw.png"))
plt.show()

plot_mean_and_variance(
    X_raw, grilla, show_std1=True, show_std2=True,
    title="Media y varianza funcional — Escenario B",
    save_path=str(PATHS["out_report"] / "03_media_varianza_raw.png"))
plt.show()

In [ ]:
# Curva observada vs verdadera: dimensiona el ruido que el modelo NO debe predecir
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4), sharey=True)
for ax, i in zip(axes, [0, T // 2, T - 1]):
    ax.plot(grilla, X_raw[i], ".", color="0.65", ms=3, label="observada (con ruido)")
    ax.plot(grilla, X_true[i], color="#c0392b", lw=1.6, label="verdadera $X_t(\\tau)$")
    ax.set_title(rf"$t={i+1}$", fontsize=10); ax.set_xlabel(r"$\tau$")
axes[0].set_ylabel(r"$X_t(\tau)$"); axes[0].legend(fontsize=8)
fig.suptitle("Curva verdadera vs datos observados — el error se mide contra la primera",
             fontsize=12)
fig.tight_layout()
fig.savefig(PATHS["out_report"] / "04_curva_vs_datos.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.6 Persistencia de curvas

Se guardan las dos matrices con nombres distintos (`X_curves.npy` y
`X_curves_true.npy`) para que no puedan confundirse aguas abajo.

In [ ]:
_p = guardar_curvas(PATHS, X_raw, grilla, X_true=X_true)
for clave, ruta in _p.items():
    print(f"[functional] {clave:12s} → {ruta.name}")

### 2.7 Partición temporal

Todo objeto **estimado a partir de los datos** —selección GCV de la base, FPCA,
estandarizador— se ajusta sólo con $\{1,\dots,T_0\}$ y se aplica al bloque de
prueba mediante `transform`.

In [ ]:
T0 = int(np.floor(PROP_TRAIN * T))
assert 10 < T0 < T, f"T0={T0} fuera de rango para T={T}."

idx_train, idx_test = np.arange(0, T0), np.arange(T0, T)
X, X_train, X_test = X_raw, X_raw[idx_train], X_raw[idx_test]

print(f"entrenamiento : t ∈ [1, {T0}]      → {X_train.shape}")
print(f"prueba        : t ∈ [{T0+1}, {T}]  → {X_test.shape}")
print(f"proporción    : {T0/T:.1%} / {1 - T0/T:.1%}")

## 3. Representación funcional B-spline

### 3.1 Barrido GCV sobre `(n_basis, order)`

El GCV **sugiere**; la elección es del analista y se declara en 3.2.

In [ ]:
N_BASIS_RANGE = range(2, min(30, T0 // 2))   # acotado por T0
ORDER_RANGE   = range(2, 5)

registros = []
for orden in ORDER_RANGE:
    for nb in N_BASIS_RANGE:
        if nb < orden:
            continue
        try:
            fr_tmp = FunctionalRepresentation(method="bspline", n_basis=nb, order=orden)
            TH_tmp = fr_tmp.fit_transform(X_train, grilla)     # sólo train
            X_rec  = fr_tmp.reconstruct(TH_tmp)

            L_i    = X_train.shape[1]
            sse_c  = np.sum((X_train - X_rec) ** 2, axis=1)
            ss_tot = np.sum((X_train - X_train.mean(axis=0, keepdims=True)) ** 2)
            gcv_c  = (L_i * sse_c / (L_i - nb) ** 2 if L_i > nb
                      else np.full_like(sse_c, np.nan))
            registros.append({
                "n_basis": nb, "order": orden,
                "var_retained": 1.0 - sse_c.sum() / ss_tot,
                "rmse_mean": np.sqrt(sse_c / L_i).mean(),
                "rmse_max":  np.sqrt(sse_c / L_i).max(),
                "gcv_mean":  float(np.mean(gcv_c)),
            })
        except Exception as e:
            print(f"  [SKIP] n_basis={nb}, order={orden}: {e}")

sel_df   = pd.DataFrame(registros)
best_row = sel_df.dropna(subset=["gcv_mean"]).nsmallest(1, "gcv_mean").iloc[0]
nb_best, ord_best = int(best_row["n_basis"]), int(best_row["order"])

display(sel_df.style
    .format({"var_retained": "{:.4%}", "rmse_mean": "{:.6f}",
             "rmse_max": "{:.6f}", "gcv_mean": "{:.6f}"})
    .background_gradient(subset=["gcv_mean"], cmap="YlOrRd_r")
    .background_gradient(subset=["var_retained"], cmap="YlGn"))

print(f"\nGCV mínimo → n_basis={nb_best}, order={ord_best}  "
      f"(var retenida {best_row['var_retained']:.4%})")

plot_seleccion_basis(sel_df, nb_best, ord_best,
                     save_path=str(PATHS["out_report"] / "05_seleccion_basis.png"))
plt.show()

### 3.2 `[CONFIG]` Base elegida y ajuste

`center=False` es imprescindible: con `center=True` la reconstrucción es un
mapa **afín**, y la función media contaminaría la base recuperada por
`base_en_grilla`, la matriz de Gram y las autofunciones.

In [ ]:
NB_ELEGIDO  = nb_best     # ← decisión del analista
ORD_ELEGIDO = ord_best

print(f"Base elegida : n_basis={NB_ELEGIDO}, order={ORD_ELEGIDO}")
print(f"Sugerido GCV : n_basis={nb_best}, order={ord_best}"
      + ("   (coinciden)" if (NB_ELEGIDO, ORD_ELEGIDO) == (nb_best, ord_best)
         else "   ← DIFIERE de la sugerencia; justificar en la tesis"))

fr = FunctionalRepresentation(method="bspline", n_basis=NB_ELEGIDO,
                              order=ORD_ELEGIDO, center=False)
fr.fit(X_train, grilla)                       # sólo train
THETA       = fr.transform(X, grilla)         # (T, K) serie completa
THETA_train = THETA[idx_train]
print(f"THETA {THETA.shape}  (train={T0}, test={T - T0})")

plot_fts_functional(
    X, grilla, fr=fr, highlight_idx=highlight_idx, separator_every=5,
    title=f"FAR(1) — repr. B-spline (n_basis={NB_ELEGIDO}, order={ORD_ELEGIDO})",
    save_path=str(PATHS["out_report"] / "06_fts_funcional_bspline.png"))
plt.show()

guardar_representacion(PATHS, fr, THETA,
    extra={"T0": int(T0), "prop_train": float(PROP_TRAIN), "ajustado_en": "train",
           "center": bool(fr.center), "n_basis": int(NB_ELEGIDO),
           "order": int(ORD_ELEGIDO), "n_basis_gcv": nb_best, "order_gcv": ord_best})
print("[functional] functional_representation.pkl + theta.csv + fr_config.json")

### 3.3 FPCA generalizado en $L^2$ y diagnóstico

In [ ]:
Phi  = base_en_grilla(fr, THETA.shape[1])        # (G, K)
fpca = FPCA_L2().fit(THETA_train, Phi, grilla)

_ver = fpca.verificar(THETA_train, fr=fr)
print("Verificación FPCA_L2 (entrenamiento):")
for k, v in _ver.items():
    print(f"  {k:34s} = {v:.3e}" if isinstance(v, float) else f"  {k:34s} = {v}")

# cond(W) amplifica el ruido de las direcciones de menor autovalor, que son las
# que después se truncan. Se registra para comparar escenarios con distinto K.
cond = _ver["cond_W"]
nota = ("← MUY ALTO: reduzca n_basis" if cond > 1e10 else
        "← alto: vigile las componentes menores" if cond > 1e6 else "(sano)")
print(f"\ncond(W) = {cond:.3e}  {nota}")

assert _ver["todo_ok"], ("Las identidades del FPCA generalizado no se cumplen. "
                         "Si falla err_linealidad_reconstruct_rel, revise center=False.")

In [ ]:
# Varianza ≠ dinámica: una FPC de varianza baja puede tener AR fuerte y ser útil
# para pronóstico, y una de varianza alta puede ser ruido temporal.
K   = fpca.evals.size
S_a = (THETA_train - fpca.mu_theta) @ (fpca.W @ fpca.B_full)     # (T0, K)
ar1 = (S_a[1:] * S_a[:-1]).sum(0) / np.clip((S_a[:-1] ** 2).sum(0), 1e-12, None)

VAR_TARGET  = 0.95
M_SUGERIDO  = fpca.seleccionar_M(VAR_TARGET)

display(pd.DataFrame({
    "componente": np.arange(1, K + 1), "autovalor": fpca.evals,
    "var_ratio": fpca.var_ratio, "var_acum": fpca.var_cum, "ar1_propio": ar1,
}).head(min(15, K)).style.format(
    {"autovalor": "{:.4e}", "var_ratio": "{:.4%}",
     "var_acum": "{:.4%}", "ar1_propio": "{:+.3f}"})
    .background_gradient(subset=["var_ratio"], cmap="YlGn")
    .background_gradient(subset=["ar1_propio"], cmap="coolwarm", vmin=-1, vmax=1))

print(f"\nK disponibles: {K}   ·   sugerencia (var ≥ {VAR_TARGET:.0%}): M = {M_SUGERIDO}")

plot_fpca_scree(fpca.evals, fpca.var_cum, M_SUGERIDO, var_target=VAR_TARGET,
                save_path=str(PATHS["out_report"] / "07_fpca_scree.png"))
plt.show()

### 3.4 `[CONFIG]` Componentes FPCA retenidas

In [ ]:
# M_FPCA se declara arriba, en la celda [BARRIDO] de §1.1, porque forma parte
# del EXPERIMENT_ID y las rutas se arman antes de llegar aquí. Esta celda sólo
# lo consume y lo contrasta con la regla del 95 % (M_SUGERIDO, de §3.3).
assert 1 <= M_FPCA <= fpca.evals.size, f"M_FPCA fuera de [1, {fpca.evals.size}]."

print(f"M(95 %) según la regla : {M_SUGERIDO}   (K disponible = {fpca.evals.size})")
print(f"M_FPCA de esta corrida : {M_FPCA}"
      + ("   (coincide con la regla)" if M_FPCA == M_SUGERIDO
         else "   ← DIFIERE de la regla: es un punto del barrido"))
fpca.set_M(int(M_FPCA))
M_fpca = fpca.M

Psi_grid, mu_grid = fpca.Psi_grid, fpca.mu_grid
SCORES       = fpca.transform(THETA)       # (T, M) — base ajustada en train
SCORES_train = SCORES[idx_train]
SCORES_test  = SCORES[idx_test]

print(f"M = {M_fpca}   var. explicada = {fpca.var_cum[M_fpca-1]:.4%}")
print(f"[train] max|media ξ| = {np.abs(SCORES_train.mean(0)).max():.2e}   (≈ 0)")
print(f"[test]  max|media ξ| = {np.abs(SCORES_test.mean(0)).max():.3f}")
print(f"[test]  var ξ / λ    = "
      f"{np.array2string(SCORES_test.var(0, ddof=1) / fpca.lambdas, precision=3)}")

### 3.5 ¿Observa el modelo la dirección de conmutación?

Idéntica a la de la corrida 18, y por el mismo motivo: la conmutación la
gobierna $z_{t-1}=\langle Y_{t-1},e\rangle$ y el modelo sólo ve los $M$ scores
retenidos. Con $e\propto\sin 2\pi\tau$ la carga está casi toda en la **segunda**
componente, de modo que la predicción es:

| $M$ | ¿Ve la conmutación? | Predicción |
|---|---|---|
| 1 | no | no distingue las ramas; predice la mezcla como si fuera unimodal |
| $\ge 2$ | sí | puede separar las modas |

`fraccion_e_explicada` $=\sum_{k\le M}\langle e,\phi_k\rangle^2$ es la cifra que
decide de qué lado cae este punto del barrido, y `_04 §11` la cruza con el MISE.


In [ ]:
_e   = salida.internos["direccion_estado"]
_w   = salida.internos["pesos_cuadratura"]
_PHI = Psi_grid                                   # (G, M)

cargas = np.array([float(np.sum(_w * _e * _PHI[:, k]))
                   for k in range(_PHI.shape[1])])
frac_e = float(np.sum(cargas ** 2))               # <= 1 por Bessel

alineacion = pd.DataFrame({
    "fpc":       np.arange(1, _PHI.shape[1] + 1),
    "carga_e":   cargas,
    "carga_e2":  cargas ** 2,
    "var_ratio": fpca.var_ratio[:_PHI.shape[1]],
    "retenida":  1,
})
alineacion.to_csv(PATHS["out_report"] / "10_alineacion_conmutacion.csv", index=False)
print(alineacion.to_string(index=False,
      formatters={"carga_e": "{:+.4f}".format, "carga_e2": "{:.4f}".format,
                  "var_ratio": "{:.2%}".format}))

FRAC_E_EXPLICADA = frac_e
print(f"\nfraccion_e_explicada (M={M_fpca}) = {frac_e:.4f}")
if frac_e < 0.25:
    print("\n[AVISO] Con este M el modelo NO observa la dirección que gobierna\n"
          "        la conmutación: es el punto BAJO del barrido y su fracaso es\n"
          "        un resultado, no un error.")
else:
    print("\n[OK] La dirección de conmutación está en el espacio retenido.")


## 4. Datasets AR($p$) sobre los scores

### 4.1 Estandarización

El estandarizador se ajusta **sólo con train** y registra `n_ajuste` para que
esa disciplina sea auditable desde el artefacto y no una promesa del notebook.

In [ ]:
scores_standardizer = DataStandardizer(method="zscore_column", ddof=0)
scores_standardizer.fit(SCORES_train, etiqueta=f"train[1:{T0}]")

_chk = scores_standardizer.verificar_ajuste(T0)
print(f"[holdout] ajustado con {_chk['n_ajuste']} filas = T0 "
      f"({_chk['etiqueta_ajuste']}) → ok={_chk['ajuste_ok']}")

SCORES_STD       = scores_standardizer.transform(SCORES)
SCORES_STD_train = SCORES_STD[idx_train]
SCORES_STD_test  = SCORES_STD[idx_test]

print(f"\nSCORES_STD {SCORES_STD.shape}")
print(f"  [train] max|media| = {np.abs(SCORES_STD_train.mean(0)).max():.2e}  (≈0)")
print(f"  [train] max|std-1| = {np.abs(SCORES_STD_train.std(0) - 1).max():.2e}  (≈0)")
print(f"  [test]  media = {np.array2string(SCORES_STD_test.mean(0), precision=3)}")
print(f"  [test]  std   = {np.array2string(SCORES_STD_test.std(0),  precision=3)}")

guardar_estandarizador(PATHS, scores_standardizer)
_res = guardar_fpca(PATHS, fpca, SCORES, SCORES_STD=SCORES_STD,
                    meta_extra={"T0": int(T0)})
print(f"\n[functional] artefactos FPCA · cond_W = {_res['meta']['cond_W']:.3e}")

plot_diagnostico_estandarizacion(
    SCORES_train, SCORES_STD_train, np.arange(1, M_fpca + 1),
    labels=("Scores ξ (escala λ)", "Scores ξ estandarizados"),
    title="estadísticas por componente FPCA",
    save_path=str(PATHS["out_report"] / "08_diagnostico_estandarizacion.png"))
plt.show()

### 4.2 Diagnóstico de rezagos (sólo train)

In [ ]:
N_LAGS_MAX = 3
T_theta, K_total = SCORES_STD_train.shape

def _spearman(Y, Xm):
    """Spearman columna a columna vía rangos (pandas, sin scipy)."""
    Yc = pd.DataFrame(Y).rank().to_numpy(); Yc = Yc - Yc.mean(0)
    Xc = pd.DataFrame(Xm).rank().to_numpy(); Xc = Xc - Xc.mean(0)
    return (Yc.T @ Xc) / np.outer(np.sqrt((Yc**2).sum(0)), np.sqrt((Xc**2).sum(0)))

corr_p = np.zeros((K_total, K_total * N_LAGS_MAX))
corr_s = np.zeros_like(corr_p)
col_labels = []
y_block = SCORES_STD_train[N_LAGS_MAX:, :]

for lag in range(1, N_LAGS_MAX + 1):
    x_block = SCORES_STD_train[N_LAGS_MAX - lag : T_theta - lag, :]
    sp = _spearman(y_block, x_block)
    for j in range(K_total):
        c = (lag - 1) * K_total + j
        for k in range(K_total):
            corr_p[k, c] = np.corrcoef(y_block[:, k], x_block[:, j])[0, 1]
        corr_s[:, c] = sp[:, j]
        col_labels.append(rf"$\xi_{{t-{lag},{j+1}}}$")

row_labels = [rf"$\xi_{{t,{k+1}}}$" for k in range(K_total)]
band = 1.96 / np.sqrt(len(y_block))

for M_corr, nombre, arch in ((corr_p, "Pearson", "09a"), (corr_s, "Spearman", "09b")):
    plot_rezagos_heatmap(
        M_corr, col_labels, row_labels,
        title=f"{nombre} — respuesta($t$) vs rezagos 1..{N_LAGS_MAX}",
        n_lags_max=N_LAGS_MAX, K_total=K_total, band=band, vclip=0.6,
        save_path=str(PATHS["out_report"] / f"{arch}_rezagos_{nombre.lower()}.png"))
    plt.show()

### 4.3 `[CONFIG]` Orden AR y construcción de los datasets

Los rezagos del primer origen de prueba vienen del final del bloque de
entrenamiento: son observaciones pasadas disponibles en cada origen, de modo
que su uso es el condicionamiento de la predicción a $h=1$, no fuga.

In [ ]:
N_LAGS        = 1
COMPONENT_IDX = list(range(SCORES_STD.shape[1]))   # base-0; los nombres usan idx+1

n_components = len(COMPONENT_IDX)
n_train_eff  = T0 - N_LAGS
n_test_eff   = T - T0

assert T0 > N_LAGS and len(set(COMPONENT_IDX)) == n_components

cov_names = [f"fpc_{COMPONENT_IDX[j] + 1}_lag{lag}"
             for lag in range(1, N_LAGS + 1)
             for j in range(n_components)]

print(f"componentes : {n_components} → índices {COMPONENT_IDX}")
print(f"N_LAGS      : {N_LAGS}   ·   p = {len(cov_names)} covariables")
print(f"n_train_eff : {n_train_eff}   n_test_eff : {n_test_eff}")
print(f"cov_names   : {cov_names}")

In [ ]:
SCORES_sel = SCORES_STD[:, COMPONENT_IDX]

def _dataset_bloque(k, t_ini, t_fin):
    """Respuesta en t ∈ [t_ini, t_fin) y predictores en t-1 … t-N_LAGS."""
    t_idx  = np.arange(t_ini, t_fin)
    X_cols = np.hstack([SCORES_sel[t_idx - lag, :] for lag in range(1, N_LAGS + 1)])
    return pd.DataFrame(np.column_stack([SCORES_sel[t_idx, k], X_cols]),
                        columns=[f"fpc_{COMPONENT_IDX[k] + 1}"] + cov_names)

dfs_train = {k: _dataset_bloque(k, N_LAGS, T0) for k in range(n_components)}
dfs_test  = {k: _dataset_bloque(k, T0,     T)  for k in range(n_components)}

manifest = {
    "scores_scale":  "standardized_zscore_ddof0",
    "n_components":  n_components,
    "n_lags":        int(N_LAGS),
    "component_idx": [int(i) for i in COMPONENT_IDX],
    "cov_names":     cov_names,
    "T": int(T), "T0": int(T0), "prop_train": float(PROP_TRAIN),
    "n_train_eff": int(n_train_eff), "n_test_eff": int(n_test_eff),
    "ajuste_en": "train",
}
guardar_datasets_ar(PATHS, dfs_train, dfs_test, manifest)
print(f"[functional] {2*n_components} datasets + datasets_manifest.json")
for k in range(n_components):
    print(f"  fpc_{COMPONENT_IDX[k]+1}: train {dfs_train[k].shape} · test {dfs_test[k].shape}")

## 5. `[CONFIG]` Hiperparámetros y configuración MCMC

Esto es el **contrato con MATLAB**: `psbp_fd_iteracion.m` lee estos valores del
JSON y no los tiene escritos a mano, incluida `seed_base`.

Ojo con la doble acepción de `M`: aquí, dentro de `mcmc_config`, es el tamaño
de la grilla de localización $G^*$ del stick-breaking, **no** el número de
componentes FPCA. `N` es el truncamiento del número de átomos.

In [ ]:
MCMC_CONFIG = {"nsim": 2000, "burn": 500, "N": 35, "M": 35}
N_CHAINS    = 3

print(f"MCMC_CONFIG : {MCMC_CONFIG}")
print(f"N_CHAINS    : {N_CHAINS} cadenas por componente "
      f"→ {N_CHAINS * n_components} jobs en MATLAB")
print(f"Draws posteriores por score: "
      f"({MCMC_CONFIG['nsim']} - {MCMC_CONFIG['burn']}) × {N_CHAINS} = "
      f"{(MCMC_CONFIG['nsim'] - MCMC_CONFIG['burn']) * N_CHAINS}")

In [ ]:
# Priors globales
HP_GLOBAL = {"atau": 2.0, "btau": 0.5, "ag": 2.0, "bg": 0.5,
             "mumu": 0.0, "taumu": 1.0, "pwj": 0.5}

# Priors por tipo de covariable: (apij, bpij, mupsij, taupsij)
HP_BY_TYPE = {
    "own_lag1":  (9.0, 1.0, 0.0, 1.0),   # E[pi] = 0.90 — el propio rezago 1
    "cross_lag": (1.0, 1.0, 0.0, 1.0),   # E[pi] = 0.50 — los cruzados
}

def _clasificar(nombre, k_modelo):
    return ("own_lag1" if nombre == f"fpc_{COMPONENT_IDX[k_modelo] + 1}_lag1"
            else "cross_lag")

HYPERPARAMS_LIST = []
for k in range(n_components):
    tipos = [_clasificar(nm, k) for nm in cov_names]
    vals  = np.array([HP_BY_TYPE[t] for t in tipos], dtype=float)   # (p, 4)
    HYPERPARAMS_LIST.append({**HP_GLOBAL,
        "apij": vals[:, 0], "bpij": vals[:, 1],
        "mupsij": vals[:, 2], "taupsij": vals[:, 3]})

for k in range(n_components):
    hp = HYPERPARAMS_LIST[k]
    print(f"\nComponente k={k}  (fpc_{COMPONENT_IDX[k]+1})")
    print(f"  {'variable':<22} {'tipo':<11} {'apij':>6} {'bpij':>6} {'E[pi]':>7}")
    for j, nm in enumerate(cov_names):
        a, b = hp["apij"][j], hp["bpij"][j]
        marca = "  ◄" if _clasificar(nm, k) == "own_lag1" else ""
        print(f"  {nm:<22} {_clasificar(nm, k):<11} {a:>6.1f} {b:>6.1f} "
              f"{a/(a+b):>7.3f}{marca}")

In [ ]:
hp_artifact = {
    "global":       HP_GLOBAL,
    "by_type":      HP_BY_TYPE,
    "mcmc_config":  MCMC_CONFIG,
    "n_iter":       N_CHAINS,
    "seed_scheme":  "seed_base + chain*9973 + k*31",
    # Cadena "B", no entero. El .m no lo lee ---arma el EXPERIMENT_ID con su
    # propio ESCENARIO_ID y un %s--- pero el artefacto debe registrar el
    # identificador tal cual es, o deja de emparejar con el nombre del directorio.
    "escenario_id": ESCENARIO_ID,
    "replica_id":   int(REPLICA_ID),
    "seed_base":    int(SEED),     # MATLAB la lee de aquí (ya no está hardcodeada)
    "scores_scale": "standardized_zscore_ddof0",
    "partition": {
        "T": int(T), "T0": int(T0), "prop_train": float(PROP_TRAIN),
        "n_train_eff": int(n_train_eff), "n_test_eff": int(n_test_eff),
        "train_files": [f"dataset_fpc_{COMPONENT_IDX[k]+1}_train.csv"
                        for k in range(n_components)],
        "test_files":  [f"dataset_fpc_{COMPONENT_IDX[k]+1}_test.csv"
                        for k in range(n_components)],
    },
    "hyperparams_list": [
        {"component_k": k, "fpc_idx": int(COMPONENT_IDX[k] + 1),
         "hyperparams": {key: (v.tolist() if isinstance(v, np.ndarray) else v)
                         for key, v in HYPERPARAMS_LIST[k].items()}}
        for k in range(n_components)
    ],
}
guardar_hiperparametros(PATHS, hp_artifact)
print(f"✓ hyperparameters.json → {PATHS['out_artefact']}")

## 6. Configuración de evaluación, líneas base y verificación del contrato

`objetivo_evaluacion` y `modo_residuo` se declaran aquí porque cambian el
significado de toda la evaluación y no deben quedar como una decisión implícita
del notebook `_04`.

In [ ]:
eval_config = {
    "scheme":      "holdout_temporal",
    "T": int(T), "T0": int(T0), "prop_train": float(PROP_TRAIN),
    "horizons":    [1],
    "n_lags":      int(N_LAGS),
    "scores_scale": "standardized_zscore_ddof0",
    # Contra QUÉ se mide el error y con qué banda. Declarado explícitamente:
    #   curva_verdadera + modo_residuo "ninguno" ⇒ la banda cubre la curva
    #   PROYECTADA sobre las M autofunciones y se contrasta con X_t(tau). Lo
    #   que queda fuera es truncamiento FPCA puro, sin el ruido de medición.
    "objetivo_evaluacion": "curva_verdadera",
    "modo_residuo":        "ninguno",
    "nivel_credibilidad":  0.95,
    "ventana_movil": {"w": [10, 20, 40], "paso": 1, "solapadas": True},
    "metrics_scores": ["RMSE", "R2", "razon_dispersion"],
    "metrics_curvas": ["MISE", "RMSE_funcional"],
    "metrics_dist":   ["CRPS", "energy_score", "cobertura_95", "PIT"],
    # ── Estratificación por el estado VERDADERO del generador ──────────────
    # Como en la corrida 18: el estrato NO coincide con la partición train/test.
    "estratificacion": {
        "variable":  "ambiguo",
        "fuente":    "reports/<EXPERIMENT_ID>/10_estado_signo_tendencia.csv",
        "metodo":    "estado discreto del generador (no se cuantiliza)",
        "n_estratos": 2,
        "etiquetas": ["determinista", "ambiguo"],
        "nota": ("`bimodal` = ambiguo Y con separación mayor que la sd de la "
                 "innovación. La separación CRECE con t, de modo que la fracción "
                 "bimodal es mayor en el bloque de prueba que en el de "
                 "entrenamiento: ése es el rasgo del escenario."),
    },
    # ── Referencias ORACLE del generador ───────────────────────────────────
    # Son las cifras contra las que _04 §9.1 contrasta la predictiva.
    "forma_predictiva": {
        "r2_lineal_fuera_de_muestra":  DIAG["r2_lineal_fuera_de_muestra"],
        "r2_oraculo_fuera_de_muestra": DIAG["r2_oraculo_fuera_de_muestra"],
        "brecha_oraculo_lineal":       DIAG["brecha_oraculo_lineal"],
        "r2_lineal_destendenciado":    DIAG["r2_lineal_destendenciado"],
        "r2_oraculo_destendenciado":   DIAG["r2_oraculo_destendenciado"],
        "brecha_oraculo_lineal_destendenciada":
            DIAG["brecha_oraculo_lineal_destendenciada"],
        "sarle_oraculo_ambiguos_train":     DIAG["sarle_oraculo_ambiguos_train"],
        "sarle_oraculo_ambiguos_test":      DIAG["sarle_oraculo_ambiguos_test"],
        "sarle_oraculo_deterministas_test": DIAG["sarle_oraculo_deterministas_test"],
        "sarle_referencia_uniforme":        DIAG["sarle_referencia_uniforme"],
        "sarle_referencia_gaussiana":       DIAG["sarle_referencia_gaussiana"],
        "n_regimenes":               DIAG["n_regimenes"],
        "modas_efectivas_media":     DIAG["modas_efectivas_media"],
        "modas_efectivas_maxima":    DIAG["modas_efectivas_maxima"],
        "separacion_modas_tendencia_maxima": DIAG["separacion_modas_tendencia_maxima"],
        "separacion_total_en_sd_T0":        DIAG["separacion_total_en_sd_T0"],
        "separacion_total_en_sd_T":         DIAG["separacion_total_en_sd_T"],
    },
    # ── La tendencia, para que viaje con los datos ─────────────────────────
    # No es decorativa: el centrado del FPCA y del estandarizador se ajustan con
    # el bloque de entrenamiento y dejan de ser válidos en el de prueba. Parte
    # del error de test es de VIGENCIA DEL CENTRADO y no de predicción.
    "tendencia": {
        "forma":              DIAG["forma_tendencia"],
        "deriva_total":       DIAG["deriva_total"],
        "inclinacion":        DIAG["inclinacion"],
        "b_en_T0":            DIAG["b_en_T0"],
        "b_en_T":             DIAG["b_en_T"],
        "desfase_train_test": DIAG["desfase_train_test"],
        "desfase_en_sd":      DIAG["desfase_en_sd"],
        "estacionario":       False,
        "advertencia": ("El proceso NO es estacionario. Parte del error de test es "
                        "de vigencia del centrado, no de capacidad predictiva; "
                        "contrastar r2_oraculo con r2_oraculo_destendenciado."),
    },
    # Cuánto del mecanismo ve el modelo con ESTE M (§3.5).
    "fraccion_e_explicada": float(FRAC_E_EXPLICADA),
}
guardar_config_evaluacion(PATHS, eval_config)
print("[out_artefact] eval_config.json")
for k, v in eval_config.items():
    print(f"  {k:22s}: {v}")

In [ ]:
# Líneas base a h=1: el piso que el PSBP-FD debe superar.
baselines_df = tabla_baselines(SCORES_STD, T0, estandarizador=scores_standardizer,
                               fpca=fpca, X_obs=X_true, tau=grilla, h=1)
baselines_df.to_csv(PATHS["out_report"] / "30_baselines_test.csv")
display(baselines_df.style.format("{:.4f}", na_rep="—")
        .set_caption("Líneas base — bloque de prueba, h=1, contra la curva VERDADERA"))

In [ ]:
informe = verificar_contrato(PATHS)
print(f"contrato_ok = {informe['contrato_ok']}   "
      f"(M={informe['M']}, K={informe['K']}, T0={informe['T0']}, "
      f"n_components={informe['n_components']})")
print(f"estandarizador ajustado con {informe.get('estandarizador_n_ajuste')} filas (T0={informe['T0']})")
print(f"verificación FPCA: todo_ok = {informe['verificacion_fpca']['todo_ok']}")
if not informe["contrato_ok"]:
    print("\nPROBLEMAS:")
    for p in informe.get("problemas", []):
        print(f"  - {p}")

print(f"\n{'='*66}\nListo. Siguiente paso, en MATLAB desde esta carpeta:\n"
      f"  >> psbp_fd_iteracion\n"
      f"EXPERIMENT_ID = {EXPERIMENT_ID}\n{'='*66}")